In [13]:
import os
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [14]:
# Definir rutas relativas a partir del directorio base
BASE_DIR = Path("..") 
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATASET_DIR = DATA_RAW_DIR / "waste_classification"
MODELS_DIR = BASE_DIR / "models"

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

# Crear carpeta de modelos si no existe
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Verificación estricta de la existencia de los datos
if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    print(f"⚠️ ¡Atención! No se encontró la estructura esperada en {DATASET_DIR}")
    print("Asegúrate de colocar tus carpetas 'train' y 'test' manualmente ahí.")
else:
    print(f"✅ Directorios de datos detectados correctamente en:\n - {TRAIN_DIR}\n - {TEST_DIR}")

✅ Directorios de datos detectados correctamente en:
 - ../data/raw/waste_classification/train
 - ../data/raw/waste_classification/test


In [15]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# 1. Generador de Entrenamiento (CON Data Augmentation)
train_datagen = ImageDataGenerator(
    rotation_range=30,      
    width_shift_range=0.2,  
    height_shift_range=0.2, 
    zoom_range=0.3,         
    horizontal_flip=True    
)

# 2. Generador de Prueba/Validación (SIN alteraciones, solo normalizado matemático)
test_datagen = ImageDataGenerator()

print("Cargando set de ENTRENAMIENTO...")
train_gen = train_datagen.flow_from_directory(
    str(TRAIN_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True # Fundamental mezclar para entrenar
)

print("Cargando set de PRUEBA/VALIDACIÓN...")
val_gen = test_datagen.flow_from_directory(
    str(TEST_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # No se mezcla para poder evaluar ordenadamente
)

NUM_CLASSES = train_gen.num_classes
print(f"\nTotal de clases detectadas: {NUM_CLASSES}")
print(f"Diccionario de clases: {train_gen.class_indices}")

Cargando set de ENTRENAMIENTO...
Found 22864 images belonging to 5 classes.
Cargando set de PRUEBA/VALIDACIÓN...
Found 5812 images belonging to 5 classes.

Total de clases detectadas: 5
Diccionario de clases: {'crushed_metal': 0, 'crushed_plastic': 1, 'metal': 2, 'no_reciclable': 3, 'plastic': 4}


In [16]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Large
# Configurar el modelo base pre-entrenado (MobileNetV3)
print("Descargando/Cargando modelo MobileNetV3(esto puede tardar un poco la primera vez)...")

# Instanciamos el modelo MobileNetV3
base_model = MobileNetV3Large(
    weights='imagenet',       # Usamos el conocimiento previo de ImageNet
    include_top=False,        # ¡Crucial! Quitamos la capa de clasificación final (la "cabeza")
    input_shape=(224, 224, 3) # Definimos que entrarán imágenes de 224x224 con 3 canales (RGB)
)

# --- CONGELAMIENTO DE CAPAS ---
# "Congelamos" el modelo base. Esto significa que sus pesos (números internos)
# NO cambiarán durante la primera fase del entrenamiento.
# Esto nos permite usarlo solo como un "extractor de características" ultra avanzado.
base_model.trainable = False

print("✅ Modelo base cargado y congelado correctamente.")
print(f"   Estructura de entrada: {base_model.input_shape}")

Descargando/Cargando modelo MobileNetV3(esto puede tardar un poco la primera vez)...
✅ Modelo base cargado y congelado correctamente.
   Estructura de entrada: (None, 224, 224, 3)


In [17]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# Construcción de la nueva cabecera del modelo
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x) # Capa intermedia para aprender patrones complejos
x = Dropout(0.3)(x) # Apaga el 30% de neuronas al azar para evitar memorización (overfitting)

# Capa de salida
output = Dense(NUM_CLASSES, activation='softmax')(x)

# Ensamblamos el modelo final
model = Model(inputs=base_model.input, outputs=output)

print("Arquitectura del modelo finalizada.")
model.summary() # Muestra un resumen para verificar que todo conecta bien

Arquitectura del modelo finalizada.


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 112, 112,  │        432 │ rescaling_3[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 112, 112,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_60       │ (None, 112, 112,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        144 │ activation_60[0]… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_57 (ReLU)     │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        256 │ re_lu_57[0][0]    │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_add   │ (None, 112, 112,  │          0 │ activation_60[0]… │
│ (Add)               │ 16)               │            │ expanded_conv_pr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 112, 112,  │      1,024 │ expanded_conv_ad… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 112, 112,  │        256 │ expanded_conv_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_58 (ReLU)     │ (None, 112, 112,  │          0 │ expanded_conv_1_… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 113, 113,  │          0 │ re_lu_58[0][0]    │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 56, 56,    │        576 │ expanded_conv_1_… │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 56, 56,    │        256 │ expanded_conv_1_

 Total params: 3,120,005 (11.90 MB)

 Trainable params: 123,653 (483.02 KB)

 Non-trainable params: 2,996,352 (11.43 MB)

In [18]:
import tensorflow as tf
print("Versión de TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detectada: {gpus}")
else:
    print("⚠️ No se detectó GPU. El entrenamiento correrá en CPU (Lento).")

Versión de TensorFlow: 2.20.0
⚠️ No se detectó GPU. El entrenamiento correrá en CPU (Lento).


In [19]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

# COMPILACIÓN DEL MODELO 
# Definimos cómo va a aprender la red neuronal
print("Compilando el modelo con optimizador Adam...")

model.compile(
    optimizer=Adam(learning_rate=1e-3), # 0.001: Velocidad de aprendizaje inicial estándar
    loss='categorical_crossentropy',    # Obligatorio para clasificación de múltiples clases (>2)
    metrics=['accuracy']                # Queremos monitorear qué tan preciso es (0.0 a 1.0)
)

# ENTRENAMIENTO FITTING 
print("\nIniciando entrenamiento (Fase 1 - Cabecera)...")
print("   Nota: Esto puede tardar dependiendo de tu hardware (CPU vs GPU).")

history = model.fit(
    train_gen,               # Datos de entrenamiento (con data augmentation)
    validation_data=val_gen, # Datos de validación (para examen final de cada época)
    epochs=15,               # Número de vueltas completas al dataset
    verbose=1                # Muestra una barra de progreso animada
)

print("¡Entrenamiento fase 1 completado!")

Compilando el modelo con optimizador Adam...

Iniciando entrenamiento (Fase 1 - Cabecera)...
   Nota: Esto puede tardar dependiendo de tu hardware (CPU vs GPU).
Epoch 1/15
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 580s 401ms/step - accuracy: 0.9444 - loss: 0.1752 - val_accuracy: 0.9695 - val_loss: 0.0937
Epoch 2/15
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 557s 390ms/step - accuracy: 0.9633 - loss: 0.1100 - val_accuracy: 0.9747 - val_loss: 0.0745
Epoch 3/15
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 576s 403ms/step - accuracy: 0.9706 - loss: 0.0889 - val_accuracy: 0.9768 - val_loss: 0.0693
Epoch 4/15
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 553s 387ms/step - accuracy: 0.9737 - loss: 0.0779 - val_accuracy: 0.9780 - val_loss: 0.0695
Epoch 5/15
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 541s 379ms/step - accuracy: 0.9766 - loss: 0.0689 - val_accuracy: 0.9776 - val_loss: 0.0625
Epoch 6/15
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 543s 380ms/step - accuracy: 0.9776 - loss: 0.0656 - val_accuracy: 0.9799 - val_loss: 0.0651
Epoch 7/15
1429/1429 ━━━━━━━━━━━━━━

In [20]:
import tensorflow as tf

print("Iniciando configuración de Fine-Tuning...")

# Descongelamos todo el modelo base primero
base_model.trainable = True

# Decidimos dónde cortar: Congelaremos el 70% inferior
fine_tune_at = int(len(base_model.layers) * 0.7)

# Congelamos las capas anteriores al punto de corte
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"   Total de capas en MobileNetV3: {len(base_model.layers)}")
print(f"   Capas congeladas (70%): {fine_tune_at}")
print(f"   Capas entrenables (30%): {len(base_model.layers) - fine_tune_at}")

# RE-COMPILACIÓN (Obligatorio después de descongelar)
# ¡IMPORTANTE! Usamos un Learning Rate MUY BAJO (1e-5)
# Esto es para hacer "micro-ajustes" sin romper los pesos ya aprendidos.
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nIniciando Fase 2 de Entrenamiento (Fine-Tuning)...")
print("   Nota: Esto tomará más tiempo por época, pero mejorará la precisión.")

# Entrenamiento final
history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20 # Entrenamos por 20 épocas más
)

print("¡Entrenamiento completo finalizado!")

Iniciando configuración de Fine-Tuning...
   Total de capas en MobileNetV2: 187
   Capas congeladas (70%): 130
   Capas entrenables (30%): 57

Iniciando Fase 2 de Entrenamiento (Fine-Tuning)...
   Nota: Esto tomará más tiempo por época, pero mejorará la precisión.
Epoch 1/20
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 658s 450ms/step - accuracy: 0.9694 - loss: 0.1061 - val_accuracy: 0.9823 - val_loss: 0.0642
Epoch 2/20
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 656s 459ms/step - accuracy: 0.9788 - loss: 0.0664 - val_accuracy: 0.9837 - val_loss: 0.0625
Epoch 3/20
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 657s 460ms/step - accuracy: 0.9812 - loss: 0.0562 - val_accuracy: 0.9845 - val_loss: 0.0610
Epoch 4/20
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 644s 450ms/step - accuracy: 0.9835 - loss: 0.0521 - val_accuracy: 0.9850 - val_loss: 0.0585
Epoch 5/20
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 656s 459ms/step - accuracy: 0.9858 - loss: 0.0424 - val_accuracy: 0.9849 - val_loss: 0.0569
Epoch 6/20
1429/1429 ━━━━━━━━━━━━━━━━━━━━ 660s 462ms/step - accur

In [21]:
import os

# GUARDADO DEL MODELO 

# Aseguramos que el directorio de modelos exista
# Si un compañero borró la carpeta 'models' o no se bajó de git, esto la crea.
if not MODELS_DIR.exists():
    print(f"La carpeta {MODELS_DIR} no existía. Creándola...")
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Definimos la ruta completa
# Usamos la extensión .keras que es el estándar actual más seguro y ligero que .h5
model_save_path = MODELS_DIR / "modelo_residuos_rpi_V3_04.keras"

print(f"Guardando modelo maestro en: {model_save_path} ...")

# Guardamos el archivo
model.save(model_save_path)

# Confirmación visual con ruta absoluta para evitar confusiones sobre dónde se guardó el modelo
print(f"¡Éxito! El archivo está listo.")
print(f"   Ruta absoluta: {model_save_path.resolve()}")

Guardando modelo maestro en: ../models/modelo_residuos_rpi_V3_04.keras ...
¡Éxito! El archivo está listo.
   Ruta absoluta: /home/lordaaguakate/Documentos/Github/EntrenamientoIA/models/modelo_residuos_rpi_V3_04.keras


In [22]:
print("Iniciando conversión a TFLite...")

# Cargar el convertidor desde el modelo Keras en memoria
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Aplicar optimizaciones Cuantización
# Esto reduce el tamaño del archivo y acelera la inferencia en la Raspberry Pi
# sin sacrificar casi nada de precisión.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Realizar la conversión
tflite_model = converter.convert()

# Guardar el archivo .tflite
tflite_save_path = MODELS_DIR / "modelo_residuos_rpi_V3_04.tflite"

with open(tflite_save_path, "wb") as f:
    f.write(tflite_model)

print(f"Modelo TFLite guardado en: {tflite_save_path}")

# --- COMPARACIÓN DE TAMAÑOS ---
# Esto es meramente para mostrar en el reporte final
keras_size = os.path.getsize(model_save_path) / (1024 * 1024)
tflite_size = os.path.getsize(tflite_save_path) / (1024 * 1024)

print(f"\nReporte de Optimización:")
print(f"   Tamaño Original (.keras): {keras_size:.2f} MB")
print(f"   Tamaño Optimizado (.tflite): {tflite_size:.2f} MB")
print(f"   Reducción de tamaño: {((keras_size - tflite_size) / keras_size) * 100:.2f}% menos")

Iniciando conversión a TFLite...
INFO:tensorflow:Assets written to: /tmp/tmphrkwtwiw/assets


INFO:tensorflow:Assets written to: /tmp/tmphrkwtwiw/assets


Saved artifact at '/tmp/tmphrkwtwiw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_773')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  140091701007120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701006736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701007312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701007888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701008080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701006160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701007504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701007696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701006928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140091701009040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1400917010

W0000 00:00:1782519521.288134   32047 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1782519521.288158   32047 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-06-26 18:18:41.290725: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmphrkwtwiw
2026-06-26 18:18:41.301798: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-06-26 18:18:41.301826: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmphrkwtwiw
I0000 00:00:1782519521.402082   32047 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-06-26 18:18:41.429211: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-06-26 18:18:42.094814: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmphrkwtwiw
2026-06-26 18:18:42.282321: I tensorflow/cc/saved_model/loader.cc:471] SavedModel 

Modelo TFLite guardado en: ../models/modelo_residuos_rpi_V3_04.tflite

Reporte de Optimización:
   Tamaño Original (.keras): 32.47 MB
   Tamaño Optimizado (.tflite): 3.24 MB
   Reducción de tamaño: 90.01% menos
